# Detectivul de date — Rezolvare

Problemele tratate:
1) Valori lipsă (`venit`, `vârstă`)  
2) Vârste invalide (<0, >120)  
3) Venituri negative  
4) `venit` ca text (`"6000 EUR"`, `12.345`)  
5) `oraș` scris inconsistent (`CLUJ`, `cluj`, ` Cluj ` etc.)  
6) `gen` scris inconsistent (`f`, `Fem`, `masculin`, etc.)

> Notebook-ul include: diagnoză → plan → curățare → verificare finală.

In [ ]:
# 📦 Importuri
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)

## 1) Încărcare date

In [ ]:
# încărcare

FILE_ID = '1bm1cCwjyiPtacCMVKW7v_X7YYI0GaX8_'
path = f"https://drive.google.com/uc?id={FILE_ID}"

df_raw = pd.read_csv(path)
df_raw.head()

,id_client,vârstă,înălțime_cm,greutate_kg,venit,gen,oraș,segment,data_inscriere
0,1,39.0,177.9,76.8,2501.0,M,București,C,2021-06-06
1,2,55.0,164.8,55.6,2949.0,F,bUCUREȘTI,B,2019-06-03
2,3,25.0,170.2,68.7,2703 EUR,F,Cluj,A,2023-09-14
3,4,69.0,167.6,74.2,3415.0,F,NaN,B,2017-09-30
4,5,55.0,177.3,77.6,3997.0,M,Timișoara,A,2015-07-31


## 2) Diagnoză inițială

Ne uităm la forme, tipuri de date, lipsuri și distribuții rapide.

In [ ]:
print("(linii, coloane):", df_raw.shape)
df_raw.info()

(linii, coloane): (300, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_client       300 non-null    int64  
 1   vârstă          290 non-null    float64
 2   înălțime_cm     300 non-null    float64
 3   greutate_kg     300 non-null    float64
 4   venit           285 non-null    object 
 5   gen             298 non-null    object 
 6   oraș            294 non-null    object 
 7   segment         300 non-null    object 
 8   data_inscriere  300 non-null    object 
dtypes: float64(3), int64(1), object(5)
memory usage: 21.2+ KB


In [ ]:
df_raw.describe()

,id_client,vârstă,înălțime_cm,greutate_kg
count,300.000000,290.000000,300.000000,300.000000
mean,150.500000,47.179310,169.030000,68.803667
std,86.746758,20.972933,9.145017,12.833071
min,1.000000,-9.000000,134.000000,45.000000
25%,75.750000,30.250000,162.900000,59.775000
50%,150.500000,48.000000,168.850000,68.800000
75%,225.250000,63.000000,176.025000,77.000000
max,300.000000,139.000000,194.500000,109.700000


In [ ]:
# Distribuții rapide pe categorice: oraș / gen / segment
for col in ["oraș", "gen", "segment"]:
    print(f"\n=== {col} ===")
    print(df_raw[col].value_counts(dropna=False).head(20))


=== oraș ===
oraș
Cluj          104
București      79
Timișoara      77
 timișoara      9
cluj            7
CLUJ            6
NaN             6
bUCUREȘTI       4
Cluj            3
Bucuresti       3
TIMIȘoara       2
Name: count, dtype: int64

=== gen ===
gen
F           157
M           128
Fem           4
FEM           4
m             3
NaN           2
f             1
masculin      1
Name: count, dtype: int64

=== segment ===
segment
B    129
A    106
C     65
Name: count, dtype: int64


### Observații așteptate
- `venit` are valori **lipsă** și amestec de tipuri (numere + text).
- `vârstă` are valori **lipsă** și valori **invalide** (sub 0 sau peste 120).
- `venit` are și valori **negative**.
- `oraș` și `gen` au scrieri **inconsistente** (litere mici/mari, spații, forme alternative).

## 3) Plan de curățare (rezumat)
1. **Transformăm `venit`** în numeric (scoaterea textului, separatorilor), păstrând NaN unde nu se poate converti.  
2. **Tratăm lipsurile**: `venit` → imputare cu mediană; `vârstă` → fie imputare (ex: mediană), fie eliminare (în acest exemplu: **imputăm mediană** pentru a nu pierde rânduri).  
3. **Eliminăm valori invalide**: `vârstă` < 0 sau > 120; **corectăm după imputare** dacă apar outlieri imposibili.  
4. **Eliminăm venituri negative** (nu au sens în contextul nostru).  
5. **Standardizăm `oraș`**: strip spații, lowercase, map la forme canonice cu diacritice.  
6. **Standardizăm `gen`**: mapează variantele la `F` / `M`, altceva → NaN (opțional, imputare/ștergere).

## 4) Implementare curățare

In [ ]:
df = df_raw.copy()

### 4.1) `venit` → numeric (curățare text)
Cazuri: `"6000 EUR"`, `"12.345"` ca string; NaN rămâne NaN.

In [ ]:
def to_numeric_income(x):
    if pd.isna(x):
        return np.nan
    # transformăm în string
    s = str(x).strip()
    # eliminăm 'EUR', spații, puncte folosite ca separatori de mii
    s = s.replace("EUR", "").replace("eur", "").strip()
    # înlocuim punctele dintre cifre cu nimic (separatori de mii)
    s = re.sub(r"(?<=\d)\.(?=\d{3}(\D|$))", "", s)
    # înlocuim virgula cu punct dacă apare ca separator zecimal
    s = s.replace(",", ".")
    # dacă după curățare nu e numeric, returnăm NaN
    try:
        return float(s)
    except ValueError:
        return np.nan

df["venit"] = df["venit"].apply(to_numeric_income)
df["venit"].head(10)

,venit
0,2501.0
1,2949.0
2,2703.0
3,3415.0
4,3997.0
5,3410.0
6,NaN
7,2782.0
8,3155.0
9,2865.0


### 4.2) Tratare lipsuri
- `venit` → imputăm cu **mediană** (robust la outlieri)
- `vârstă` → imputăm cu **mediană** (exemplu; alternativ se pot elimina rândurile lipsă)

In [ ]:
venit_mediana = df["venit"].median(skipna=True)
varsta_mediana = df["vârstă"].median(skipna=True)

df["venit"] = df["venit"].fillna(venit_mediana)
df["vârstă"] = df["vârstă"].fillna(varsta_mediana)

### 4.3) Eliminare valori invalide / negative
- `vârstă` < 0 sau > 120 → eliminăm
- `venit` < 0 → eliminăm

In [ ]:
cond_age_valid = (df["vârstă"] >= 0) & (df["vârstă"] <= 120)
cond_income_valid = df["venit"] >= 0

before = df.shape[0]
df = df[cond_age_valid & cond_income_valid].copy()
after = df.shape[0]
print(f"Eliminate ca invalide/negative: {before - after} rânduri")

Eliminate ca invalide/negative: 14 rânduri


### 4.4) Standardizare `oraș`
- strip spații, lowercase, apoi map la **Cluj / București / Timișoara** cu diacritice.

In [ ]:
def normalize_city(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip().lower()
    # normalizări simple
    s = s.replace("ș", "s").replace("ţ", "t").replace("ț", "t").replace("ă", "a").replace("â", "a").replace("î", "i")
    mapping = {
        "cluj": "Cluj",
        "cluj-napoca": "Cluj",
        "bucuresti": "București",
        "bucuresti sector 1": "București",
        "bucuresti sector 2": "București",
        "timisoara": "Timișoara",
        "timişoara": "Timișoara",
        "": np.nan,
        "na": np.nan,
        "none": np.nan
    }
    # eliminăm spații în plus
    s = s.strip()
    return mapping.get(s, "Cluj" if "cluj" in s else ("București" if "bucurest" in s else ("Timișoara" if "timisoar" in s else np.nan)))

df["oraș"] = df["oraș"].apply(normalize_city)
df["oraș"].value_counts(dropna=False)

,count
oraș,
Cluj,116
Timișoara,86
București,78
NaN,6


### 4.5) Standardizare `gen`
- Mapează toate variantele la `F` sau `M`.  
- Altele (gol, `None`, `NA`) → NaN (opțional: imputare / excludere ulterior).

In [ ]:
# 1. Convertim valorile în string
df["gen_clean"] = df["gen"].astype(str)

# 2. Eliminăm spațiile de la început și sfârșit
df["gen_clean"] = df["gen_clean"].str.strip()

# 3. Transformăm în litere mici
df["gen_clean"] = df["gen_clean"].str.lower()

# 4. Mapăm valorile la "F" sau "M"
gender_map = {
    "f": "F", "fem": "F", "femeie": "F", "feminin": "F", "feminină": "F",
    "m": "M", "masculin": "M", "barbat": "M", "bărbat": "M"
}
df["gen"] = df["gen_clean"].map(gender_map)

## 5) Verificare finală

In [ ]:
print("Forma după curățare:", df.shape)
display(df.info())
display(df.describe())

Forma după curățare: (286, 10)
<class 'pandas.core.frame.DataFrame'>
Index: 286 entries, 0 to 299
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_client       286 non-null    int64  
 1   vârstă          286 non-null    float64
 2   înălțime_cm     286 non-null    float64
 3   greutate_kg     286 non-null    float64
 4   venit           286 non-null    float64
 5   gen             284 non-null    object 
 6   oraș            280 non-null    object 
 7   segment         286 non-null    object 
 8   data_inscriere  286 non-null    object 
 9   gen_clean       286 non-null    object 
dtypes: float64(4), int64(1), object(5)
memory usage: 24.6+ KB


None

,id_client,vârstă,înălțime_cm,greutate_kg,venit
count,286.000000,286.000000,286.000000,286.000000,286.000000
mean,151.734266,46.846154,168.786713,68.447902,3201.954545
std,87.155071,17.150033,9.155048,12.636949,697.513872
min,1.000000,18.000000,134.000000,45.000000,1200.000000
25%,75.250000,31.250000,162.800000,59.700000,2760.500000
50%,152.500000,48.000000,168.550000,68.700000,3156.000000
75%,227.750000,63.000000,175.850000,76.975000,3681.250000
max,300.000000,74.000000,194.500000,105.500000,4975.000000


In [ ]:
# Comparație scurtă pe categorice
for col in ["oraș", "gen", "segment"]:
    print(f"\n=== {col} (după curățare) ===")
    print(df[col].value_counts(dropna=False).head(20))


=== oraș (după curățare) ===
oraș
Cluj         116
Timișoara     86
București     78
NaN            6
Name: count, dtype: int64

=== gen (după curățare) ===
gen
F      161
M      123
NaN      2
Name: count, dtype: int64

=== segment (după curățare) ===
segment
B    120
A    102
C     64
Name: count, dtype: int64


---

### Rezumat (ce am făcut și de ce)
- Am **diagnosticat**: lipsuri, tipuri amestecate, inconsistențe categorice, reguli de validare.  
- Am **curățat**: conversie `venit` la numeric, imputare mediană pentru lipsuri, eliminat valori invalide/negative, standardizat `oraș` și `gen`.  
- Am **verificat**: `info()`, `describe()`, distribuții pe categorice după curățare.  

> Acest pipeline (diagnoză → tratament → verificare) e un sablon reutilizabil pentru EDA reală.